# A measured cost makes the global fixed-row comparison concrete.

Seol Handong

The fixed-budget question in [idea.md](../../../../preliminary_research/idea.md) is

$$\boxed{\max_{\|M\|_1=R}\frac{I(M)}{\sum_{C\in\mathcal C(M)}T(|C|)}.}$$

We retain the last-zero dynamic program, but replace its illustrative cost and synthetic main inputs. Notebook 01 supplies a fitted native-SSD profile; Experiment 22 supplies dense-forward activation importance. The released selector supplies the comparison mask. Every competing mask is evaluated at its **realized** row count.

This is an offline optimum under an additive fitted model, not a measured end-to-end speedup. The archive contains text-model activations, not a new VLM evaluation. All algorithms and report calculations remain in this notebook. A normal evaluation loads available results; the full experiment and the large scaling study require separate explicit switches.

In [1]:
from pathlib import Path
from itertools import product
from time import perf_counter
from tempfile import NamedTemporaryFile
import hashlib, json, platform, sys
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter1d
from IPython.display import Markdown, display

HERE = Path.cwd()
ROOT = next(p for p in (HERE, *HERE.parents) if (p / ".gitmodules").is_file())
PROFILE = ROOT / "conclusion/python/results/01/block_estimates.csv"
MODEL = PROFILE.with_name("latency_model.json")
TRACE = ROOT / "experiments/22_predicted_lambda_trim/results/activation_traces.npz"
UPSTREAM = ROOT / "conclusion/upstream/vlm-flash"
UPSTREAM_SHA = "9023679ebfd1900fa2912c4098f51bfb9efa7bb8"
RUN_ORACLE = False
RUN_SCALING = False
MAX_TRACES_PER_SHAPE = 0  # 0: every archived trace; 32: the earlier 128-trace subset.
BUDGETS = tuple(np.arange(1, 10) / 10)
RTOL, MAX_TABLE_MIB = 1e-6, 1024
PAPER_IMPL = "native"  # Explicit; no silent downgrade to another implementation.
SCALING_SIZES = (128, 256, 512, 896, 2048, 4864, 8960, 14336, 18944)
SCALING_REPEATS = 3
assert len(set(BUDGETS)) == len(BUDGETS) and all(0 < f < 1 for f in BUDGETS)

## The row cost inherits the fitted KiB boundary without rounding it.

Read the full-precision model exported by [notebook 01](../01/01_chunk_latency.ipynb), rather than transcribing its displayed decimals or fitting a second model. The export is rejected if its CSV fingerprint is stale.

For a projection with $d$ output columns and $w$ bytes per stored weight, one input-channel row occupies $q=wd/1024$ KiB. We assume storage in the archive's floating-point dtype; the archive contains importance vectors, not serialized weight files. This assumption does not cover quantized or compressed storage.

$$T_q(r)=c_1qr+\delta\max(s_{\rm KiB},qr),\quad r>0,\qquad T_q(0)=0.$$

Equivalently, put $s=s_{\rm KiB}/q$, $b_1=qc_1$, $b_2=q(c_1+\delta)$ and $a=(b_2-b_1)s$. Then

$$T_q(r)=\begin{cases}a+b_1r,&1\le r\le s,\\b_2r,&r>s.\end{cases}$$

The real boundary $s$ is generally fractional. Only the short-run **index range** uses $h=\lfloor s\rfloor$; the intercept remains $a=\delta s_{\rm KiB}$. Rounding $s$ in the cost would define a different model.

In [2]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

model = json.loads(MODEL.read_text())
assert model["format"] == "vlm-in-flash-two-line-v1"
assert model["unit"] == {"length": "KiB", "time": "ms"}
if model["csv_sha256"] != sha256(PROFILE):
    raise ValueError("Stale latency_model.json: evaluate notebook 01 with RERUN_MEASUREMENTS=False.")
s_kib = float(model["s_kib"])
c1 = float(model["c1_ms_per_kib"])
delta = float(model["delta_ms_per_kib"])
s99 = float(model["s99_kib"])
assert np.isfinite([s_kib, c1, delta, s99]).all()
assert s_kib > 0 and c1 >= 0 and delta >= 0 and c1 + delta > 0
profile = pd.read_csv(PROFILE)
wide = profile.pivot(index="size_kib", columns="block", values="latency_ms").sort_index()
z_grid = wide.index.to_numpy(float)
np.testing.assert_array_equal(z_grid, model["measured_grid_kib"])
T_holdout = np.median(wide[model["holdout_blocks"]].to_numpy(), axis=1)
assert np.isfinite(T_holdout).all() and (T_holdout > 0).all()
display(Markdown(f"The inherited boundary is **{s_kib:g} KiB**; the short and tail slopes are "
    f"**{1000*c1:.6f}** and **{1000*(c1+delta):.6f} µs/KiB**. "
    "The same full-precision coefficients generate both the greedy candidate costs and the oracle objective."))

The inherited boundary is **207 KiB**; the short and tail slopes are **0.124258** and **0.183503 µs/KiB**. The same full-precision coefficients generate both the greedy candidate costs and the oracle objective.

In [3]:
def costs(n, s, b1, b2):
    r = np.arange(n + 1, dtype=float)
    T = b1 * r + (b2 - b1) * np.maximum(s, r)
    T[0] = 0.
    return T


def runs(mask):
    return np.flatnonzero(np.diff(np.r_[False, mask, False])).reshape(-1, 2)


def metrics(v, mask, T):
    intervals = runs(mask)
    return float(v[mask].sum()), float(T[np.diff(intervals, axis=1).ravel()].sum())


def top_r(v, R):
    mask = np.zeros(len(v), bool)
    mask[np.argsort(-v, kind="stable")[:R]] = True
    return mask

## Replace the ratio by a question with an additive answer.

For a proposed ratio $q\ge0$, define

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

Because every feasible latency is positive,

$$\boxed{F_R(q)\ge0\iff q\le\eta_R^\star.}$$

Thus one additive optimization tells us whether $q$ is attainable. This is not a sweep intended to recover every Pareto point: we are locating the zero of the value function for one specified ratio objective.

## The last zero gives an exact recurrence.

Write $Z=N-R$ and $P_j=\sum_{i<j}v_i$. Let $D_{z,r}$ be the largest value of $I-qL$ over prefixes of length $z+r$ containing exactly $z$ zeros and $r$ ones. With no zeros there is only one mask, so

$$D_{0,r}=P_r-qT(r).$$

When $z>0$, place the **last zero** after $t$ selected channels. Before it there are $z-1$ zeros and $t$ ones; after it comes one run of $r-t$ ones. The empty final run $t=r$ is allowed. Therefore

$$\boxed{D_{z,r}=\max_{0\le t\le r}\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].}$$

**Why it is exact.** Every feasible prefix has this unique decomposition. For a fixed $t$, replacing its earlier prefix by the best $D_{z-1,t}$ cannot change the final run or its cost. Conversely, an optimal earlier prefix, one zero, and that final run construct a feasible prefix. Induction on $z$ proves equality, with $F_R(q)=D_{Z,R}$. The separating zero prevents charging two adjacent pieces as separate runs.

This recurrence is valid for any run-length table. There are only $(Z+1)(R+1)$ states, but directly scanning all $t$ still takes $O((Z+1)(R+1)^2)$ time. The two-line form removes that extra factor.

## Two range maxima remove the scan over run lengths.

For the fixed zero count $z$, define $A_t=D_{z-1,t}-P_{z+t}+\lambda b_1t$ and $B_t=D_{z-1,t}-P_{z+t}+\lambda b_2t$. Here $\lambda$ is the trial ratio called $q$ in the preceding recurrence; it is unrelated to the row size $q$ above. With $h=\lfloor s\rfloor$,

$$D_{z,r}=\max\left\{\begin{aligned}
&D_{z-1,r},\\
&P_{z+r}-\lambda(a+b_1r)+\max_{\max(0,r-h)\le t<r}A_t,\\
&P_{z+r}-\lambda b_2r+\max_{0\le t<r-h}B_t.
\end{aligned}\right.$$

An empty maximum is $-\infty$. The short range is a trailing maximum; the long range is a prefix maximum. When $s<1$, there are no positive short runs. The [SciPy moving maximum](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.maximum_filter1d.html) is linear in its input length. Thus one value-only solve takes $O((N-R+1)(R+1))$ time and $O(R+1)$ working memory.

The ratio search needs values, not a traceback at every step. Only its final feasible boundary retains the full table for recovery. This saves memory traffic without restricting the mask family.

In [4]:
def bellman(v, R, q, s, b1, b2, *, retain=False):
    Z = len(v) - R
    P = np.r_[0., np.cumsum(v)]
    r = np.arange(R + 1)
    previous = P[:R + 1] - q * costs(R, s, b1, b2)
    D = np.empty((Z + 1, R + 1)) if retain else None
    if retain:
        D[0] = previous
    h = min(int(np.floor(s)), R)
    for z in range(1, Z + 1):
        prefix = P[z:z + R + 1]
        base = previous - prefix
        short = np.full(R + 1, -np.inf)
        if h:
            short[1:] = maximum_filter1d(base[:-1] + q * b1 * r[:-1],
                size=h, origin=(h - 1) // 2, mode="constant", cval=-np.inf)
        long = np.full(R + 1, -np.inf)
        long[h + 1:] = np.maximum.accumulate(base + q * b2 * r)[:R - h]
        previous = np.maximum.reduce([previous,
            prefix - q * ((b2 - b1) * s + b1 * r) + short,
            prefix - q * b2 * r + long])
        if retain:
            D[z] = previous
    return D if retain else float(previous[-1])

### Recover a mask at the final feasible ratio.

At $(z,r)$, rescan the original recurrence for a maximizing $t$, select $[z+t,z+r)$, and continue at $(z-1,t)$. At $z=0$, the remaining prefix consists of ones. Ties may produce different optimal masks. The final recovery takes another $O((N-R+1)(R+1))$ operations and retains one float64 value table. Its memory is not the total process memory.

In [5]:
def recover(v, R, q, T, D):
    P = np.r_[0., np.cumsum(v)]
    mask = np.zeros(len(v), bool)
    r = R
    for z in range(len(v) - R, 0, -1):
        t = np.arange(r + 1)
        scores = D[z - 1, :r + 1] + P[z + r] - P[z + t] - q * T[r - t]
        start = int(np.argmax(scores))
        mask[z + start:z + r] = True
        r = start
    mask[:r] = True
    return mask

## A bracket bounds the remaining optimization error.

Top-$R$ supplies a feasible lower bound $\ell_0$ and the upper bound

$$u_0=\frac{I(M_{\rm Top-R})}{b_2R},\qquad T_q(r)\ge b_2r.$$

Bisect $[\ell,u]$. A nonnegative additive optimum moves the feasible boundary to the midpoint; a negative value moves the upper boundary. Stop at $u-\ell\le\epsilon\ell_0$, then recover a mask at $\ell$. In exact arithmetic its actual ratio $u_M$ satisfies

$$\boxed{0\le1-u_M/\eta_R^\star\le\epsilon.}$$

For fractional $s>0$, use $\kappa=\max(1,T_q(1)/b_2)$ rather than an integer-boundary shortcut. At most $\lceil\log_2(\kappa/\epsilon)\rceil+1$ value calls suffice, followed by one retained-table solve. Zero importance is handled separately. The full arithmetic bound is

$$O\!\left(N\log N+(N-R+1)(R+1)\log(\kappa/\epsilon)\right).$$

The implementation uses float64, checks the recovered additive value and reports the final bracket. These are numerical consistency checks, not an interval-arithmetic certificate against rounding error. The word global refers to the declared fitted objective, not to measured SSD wall time or all possible accuracy objectives.

In [6]:
def solve(v, R, s, b1, b2, rtol=1e-6, max_table_mib=1024):
    v = np.asarray(v, dtype=float)
    assert v.ndim == 1 and np.isfinite(v).all() and (v >= 0).all()
    assert isinstance(R, (int, np.integer)) and 1 <= R <= len(v)
    assert np.isfinite([s, b1, b2]).all() and s > 0 and 0 <= b1 <= b2 and b2 > 0
    assert 0 < rtol < 1
    T = costs(len(v), s, b1, b2)
    initial = top_r(v, R)
    I, L = metrics(v, initial, T)
    lower, upper = I / L, I / (b2 * R)
    if I == 0:
        return initial, 0., 0., 0
    tolerance = rtol * lower
    kappa = max(1., T[1] / b2)
    max_calls = int(np.ceil(np.log2(kappa / rtol))) + 1
    mib = 8 * (len(v) - R + 1) * (R + 1) / 2**20
    if mib > max_table_mib:
        raise MemoryError(f"Traceback table needs {mib:.1f} MiB; limit is {max_table_mib}.")
    calls = 0
    while upper - lower > tolerance:
        q = (lower + upper) / 2
        if q == lower or q == upper or calls >= max_calls:
            raise ArithmeticError("Ratio search did not attain the requested bracket.")
        if bellman(v, R, q, s, b1, b2) >= 0:
            lower = q
        else:
            upper = q
        calls += 1
    D = bellman(v, R, lower, s, b1, b2, retain=True)
    mask = recover(v, R, lower, T, D)
    I, L = metrics(v, mask, T)
    ratio = I / L
    np.testing.assert_allclose(I - lower * L, D[-1, -1], rtol=1e-9,
                               atol=1e-9 * max(1., v.sum()))
    assert mask.sum() == R
    slack = 1e-10 * max(1., upper)
    if ratio + slack < lower or ratio > upper + slack:
        raise ArithmeticError("Recovered ratio conflicts with the numerical bracket.")
    upper = max(upper, ratio)
    if upper - ratio > rtol * ratio + slack:
        raise ArithmeticError("Recovered mask misses the requested ratio tolerance.")
    return mask, ratio, upper, calls + 1

### Enumeration checks the recurrence; it is not the main experiment.

At most nine channels are enumerated. Every positive budget is checked against an independent run counter, including zero and tied importance, equal slopes, fractional boundaries, boundaries below one row and boundaries beyond the vector. The two-row and retained-table Bellman paths must agree. No real trace sweep is executed by this check.

In [7]:
def enumerate_small(v, T):
    masks = np.array(list(product([False, True], repeat=len(v))))
    latency = np.zeros(len(masks))
    length = np.zeros(len(masks), int)
    for column in masks.T:
        latency[~column] += T[length[~column]]
        length = np.where(column, length + 1, 0)
    return masks, masks @ v, latency + T[length]

In [8]:
checks = 0
for n in (1, 5, 9):
    for v in (np.zeros(n), np.ones(n), np.random.default_rng(n).uniform(0, 4, n)):
        for boundary in (0.5, 1., 1.5, 4.25, n + 2.):
            for short_slope in (0., .25, 1.):
                T = costs(n, boundary, short_slope, 1.)
                masks, imp, delay = enumerate_small(v, T)
                for R in range(1, n + 1):
                    allowed = masks.sum(axis=1) == R
                    for q in (0., .1, 1., 10.):
                        D = bellman(v, R, q, boundary, short_slope, 1., retain=True)
                        mask = recover(v, R, q, T, D)
                        I, L = metrics(v, mask, T)
                        exact = np.max(imp[allowed] - q * delay[allowed])
                        assert mask.sum() == R
                        np.testing.assert_allclose([D[-1, -1], I-q*L,
                            bellman(v,R,q,boundary,short_slope,1.)], exact, atol=1e-9)
                        checks += 1
                    mask, lower, upper, calls = solve(v,R,boundary,short_slope,1.)
                    exact = np.max(imp[allowed] / delay[allowed])
                    assert lower <= exact + 1e-9 and exact <= upper + 1e-9
                    assert upper-lower <= 1e-6*max(lower,1.) + 1e-9

## Real activation traces replace the synthetic main inputs.

Read the existing [Experiment 22 archive](../../../../experiments/22_predicted_lambda_trim/results/activation_traces.npz). Its manifest records model revision, prompts, projection names, shapes and dtype. Importance is $v_i=\operatorname{mean}_{\rm batch,token}|x_i|$ from an unperturbed dense forward. No model is downloaded or executed here.

All archived traces are used by default. Setting `MAX_TRACES_PER_SHAPE = 32` selects the same evenly spaced per-shape subset rule used by Experiment 22. We retain every channel; this is not a crop to make the oracle easier. Mean normalization rescales the objective but not its exact optimum; the normalized float32 vector passed to upstream is also evaluated by the float64 oracle.

Traces from one prompt or layer are dependent observations. Shape- and prompt-level summaries are descriptive; thousands of budget cases do not create thousands of independent prompts.

In [9]:
def load_traces(path, cap=0):
    assert isinstance(cap, int) and cap >= 0
    with np.load(path, allow_pickle=False) as archive:
        manifest = json.loads(str(archive["manifest_json"].item()))
        assert manifest["format"] == "experiment-22-vlm-activation-traces-v1"
        traces = []
        for entry in manifest["traces"]:
            t = {k: v for k, v in entry.items() if k != "array"}
            v = np.asarray(archive[entry["array"]], dtype=np.float32).copy()
            assert v.shape == (t["n"],) and t["shape"] == f"{t['n']}x{t['d']}"
            assert np.isfinite(v).all() and (v >= 0).all()
            t["values"] = v
            traces.append(t)
    assert len(traces) == manifest["trace_count"]
    assert len({t["trace_id"] for t in traces}) == len(traces)
    chosen = []
    for shape in sorted({t["shape"] for t in traces}):
        group = [t for t in traces if t["shape"] == shape]
        indices = np.linspace(0, len(group)-1, cap, dtype=int) if cap and len(group)>cap else range(len(group))
        chosen.extend(group[i] for i in indices)
    return sorted(chosen, key=lambda t: t["trace_id"]), manifest


def importance_vector(trace):
    v = trace["values"].astype(float)
    return (v / v.mean()).astype(np.float32).astype(float) if v.sum() else v

traces, trace_manifest = load_traces(TRACE, MAX_TRACES_PER_SHAPE)
DTYPE_BYTES = {"float16": 2, "bfloat16": 2, "float32": 4}
weight_bytes = DTYPE_BYTES[trace_manifest["model_dtype"]]
shape_frame = pd.DataFrame([{k:t[k] for k in ("shape", "n", "d")} for t in traces])
shape_summary = shape_frame.groupby(["shape", "n", "d"]).size().rename("traces").reset_index()
shape_summary["KiB/row"] = weight_bytes * shape_summary.d / 1024
shape_summary["boundary (rows)"] = s_kib / shape_summary["KiB/row"]
shape_summary["cases"] = shape_summary.traces * len(BUDGETS)
display(Markdown(f"**{len(traces)}** selected traces from **{trace_manifest['model']}**, "
    f"revision `{trace_manifest['model_revision']}`, yield **{len(traces)*len(BUDGETS)}** budget cases. "
    f"The archive records **{trace_manifest['prompt_count']}** prompts; dtype is **{trace_manifest['model_dtype']}**."))
shape_summary.round(4)

**504** selected traces from **Qwen/Qwen2.5-0.5B-Instruct**, revision `7ae557604adf67be50417f59c2c2f167def9a775`, yield **4536** budget cases. The archive records **3** prompts; dtype is **bfloat16**.

,shape,n,d,traces,KiB/row,boundary (rows),cases
0,4864x896,4864,896,72,1.75,118.2857,648
1,896x128,896,128,144,0.25,828.0000,1296
2,896x4864,896,4864,144,9.50,21.7895,1296
3,896x896,896,896,144,1.75,118.2857,1296


## The released selector fixes the realized budget.

Call the pinned submodule's [`select_chunks`](../../../upstream/vlm-flash/src/vlmflash/policy.py); do not copy its greedy loop. Use Experiment 22's start/stride settings for each archived shape. Its candidate costs come from the same fitted model as the oracle, and the exclusive upper window bound is $s_{99}+1$ KiB. The upstream window quantization, stable sorting and non-overlap rules remain unchanged. This is the released **algorithm retargeted to the fitted device profile**, not a claim to reproduce its bundled AGX table or GPU runtime.

For nominal $R_0=\lfloor fN\rfloor$, obtain $M_P$ and set $R=\|M_P\|_1$. Solve the oracle, Top-$R$, and saturation tiles at this $R$. Empty upstream selections are recorded as exclusions rather than given an undefined ratio. Recompute every cost over maximal runs; `Selection.est_cost_ms` is not the shared evaluator because adjacent accepted candidates can merge.

Tiles use lengths $\lceil s\rceil$, offsets zero and half a tile, and one residual interval. Below a tile they use the best contiguous length-$R$ interval. They are a declared local baseline, not another upstream implementation.

Explicit native execution requires PyTorch and the submodule's C++/Ninja build dependencies. Initialize it from the repository root with `git submodule update --init -- conclusion/upstream/vlm-flash`. Cached evaluation needs no native build.

In [10]:
PAPER_GEOMETRY = {"896x896": (8.,8.), "896x128": (8.,8.),
                  "896x4864": (8.,8.), "4864x896": (12.,16.)}
assert {t["shape"] for t in traces} <= set(PAPER_GEOMETRY), "Declare parameters for new archive shapes."


def load_upstream():
    import subprocess
    if not (UPSTREAM / "src/vlmflash/policy.py").is_file():
        raise FileNotFoundError("Initialize conclusion/upstream/vlm-flash before RUN_ORACLE=True.")
    actual = subprocess.check_output(["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True).strip()
    if actual != UPSTREAM_SHA:
        raise ValueError("The checked-out upstream revision differs from this protocol.")
    sys.path.insert(0, str((UPSTREAM / "src").resolve()))
    import torch
    import vlmflash.policy as policy
    from vlmflash.latency import LatencyTable
    assert Path(policy.__file__).resolve().is_relative_to(UPSTREAM.resolve())
    if PAPER_IMPL == "native":
        from vlmflash._native import native, unavailable_reason
        if native() is None:
            raise RuntimeError(unavailable_reason())
    table = LatencyTable({int(z): float(c1*z + delta*max(s_kib,z)) for z in z_grid})
    runtime = dict(python=sys.version, platform=platform.platform(), processor=platform.processor(),
        numpy=np.__version__, scipy=scipy.__version__, torch=torch.__version__,
        cuda_available=torch.cuda.is_available(), paper_impl=PAPER_IMPL)
    return torch, policy, table, runtime


def paper_mask(trace, v, R, backend):
    torch, policy, table, _ = backend
    start, jump = PAPER_GEOMETRY[trace["shape"]]
    params = policy.ChunkParams(start_kb=start, end_kb=s99+1,
                               step_kb=start, jump_cap_kb=jump)
    row_kib = weight_bytes * trace["d"] / 1024
    selection = policy.select_chunks(torch.tensor(v, dtype=torch.float32), R,
        row_kib, table, params, impl=PAPER_IMPL)
    return selection.mask.detach().cpu().numpy().astype(bool)

In [11]:
def tiles(v, R, T, s):
    P = np.r_[0., np.cumsum(v)]
    if R < s:
        start = np.argmax(P[R:] - P[:-R])
        mask = np.zeros(len(v), bool)
        mask[start:start + R] = True
        return mask
    choices = []
    k, remaining = divmod(R, s)
    for offset in (0, s // 2):
        starts = np.arange(offset, len(v) - s + 1, s)
        if len(starts) < k:
            continue
        weights = P[starts + s] - P[starts]
        mask = np.zeros(len(v), bool)
        for start in starts[np.argsort(-weights, kind="stable")[:k]]:
            mask[start:start + s] = True
        if remaining:
            occupied = np.r_[0, np.cumsum(mask)]
            valid = occupied[remaining:] == occupied[:-remaining]
            if not valid.any():
                continue
            scores = np.where(valid, P[remaining:] - P[:-remaining], -np.inf)
            start = np.argmax(scores)
            mask[start:start + remaining] = True
        I, L = metrics(v, mask, T)
        choices.append((I / L, mask))
    return max(choices, key=lambda row: row[0])[1]

## A fitted optimum and a measured-lookup check answer different questions.

For a heuristic ratio $u_M$, the numerical oracle bracket $[\ell,u]$ bounds its fitted-objective gap:

$$\max(0,100(1-u_M/\ell))\le g(M)\le\max(0,100(1-u_M/u)).$$

The main comparison uses this common objective and common $R$. Report retained importance, modeled milliseconds, run count, selector time and fill ratio separately.

A second evaluator sums the **held-out** raw profile median at the recovered run lengths, linearly interpolating only inside the measured grid. If any run is below the minimum or above the maximum profiled KiB, its total lookup score is unavailable. We record out-of-range byte fractions rather than clamp or silently extrapolate. Pairwise lookup gains require both masks to be fully supported.

The fitted model itself extrapolates when a run leaves that range. In particular, long runs may exceed the native reader's 768 KiB task cap; the additive objective here does **not** simulate task splitting, six-way scheduling or queueing. A separate hardware experiment is needed for that question. Even an in-range summed lookup score is a modeled cost, not a fresh measurement of the complete mask.

In [12]:
def evaluate_mask(trace, v, mask, T):
    I, L = metrics(v, mask, T)
    spans = runs(mask)
    lengths = np.diff(spans, axis=1).ravel()
    kib = lengths * weight_bytes * trace["d"] / 1024
    supported = (kib >= z_grid[0]) & (kib <= z_grid[-1])
    lookup = float(np.interp(kib, z_grid, T_holdout).sum()) if supported.all() else None
    return dict(importance=I, retained_pct=100*I/v.sum(), fitted_ms=L,
        ratio=I/L, chunks=len(lengths), max_run_rows=int(lengths.max()),
        lookup_ms=lookup, lookup_ratio=I/lookup if lookup else None,
        out_of_range_byte_pct=100*float(kib[~supported].sum()/kib.sum()),
        lookup_supported=bool(supported.all()))


def one_case(trace, fraction, backend):
    v = importance_vector(trace)
    n = len(v)
    nominal = max(1, int(np.floor(fraction*n)))
    start = perf_counter()
    paper = paper_mask(trace, v, nominal, backend)
    paper_seconds = perf_counter()-start
    R = int(paper.sum())
    base = dict(trace_id=int(trace["trace_id"]), shape=trace["shape"], module=trace["module"],
        prompt=trace["prompt_sha256"], fraction=float(fraction), R_nominal=nominal, R=R)
    if R == 0 or v.sum() == 0:
        return dict(**base, status="empty_selection" if R == 0 else "zero_importance", records=[], masks={})
    q = weight_bytes*trace["d"]/1024
    s, b1, b2 = s_kib/q, q*c1, q*(c1+delta)
    T = costs(n,s,b1,b2)
    start = perf_counter()
    oracle, lower, upper, calls = solve(v,R,s,b1,b2,RTOL,MAX_TABLE_MIB)
    seconds = {"Global DP": perf_counter()-start, "Upstream greedy": paper_seconds}
    masks = {"Global DP": oracle, "Upstream greedy": paper}
    for name, select in (("Top-R", lambda: top_r(v,R)),
                         ("Saturation tiles", lambda: tiles(v,R,T,max(1,int(np.ceil(s)))))):
        start = perf_counter()
        masks[name] = select()
        seconds[name] = perf_counter()-start
    rows = []
    for name, mask in masks.items():
        assert mask.shape == v.shape and mask.sum() == R
        result = evaluate_mask(trace,v,mask,T)
        assert result["ratio"] <= upper + 1e-8*max(1.,upper)
        rows.append(dict(**base, method=name, **result,
            gap_lower_pct=max(0.,100*(1-result["ratio"]/lower)),
            gap_upper_pct=max(0.,100*(1-result["ratio"]/upper)),
            oracle_lower=lower, oracle_upper=upper, oracle_calls=calls,
            selector_seconds=seconds[name], fill_pct=100*R/nominal))
    return dict(**base, status="ok", records=rows,
                masks={name:np.flatnonzero(mask).tolist() for name,mask in masks.items()})

## Plan the full archive, but do not confuse a projection with a timing measurement.

The default experiment evaluates every archived trace at fractions $0.1,\ldots,0.9$. The table below estimates work at the **nominal** budgets; actual $R$ is learned only after upstream selection. It uses the earlier notebook's saved host timings and state counts. No calibration sweep runs while constructing this plan.

The projection scales the old 4,864-channel solve by states and a conservative model-dependent call count. It excludes upstream compilation, greedy selection, cache writes and host differences, and the new rolling-value implementation need not have the old timing constant. It is a planning calculation, not a promised duration or a statistical bound. The separately enabled scaling study measures complete new solves and labels its synthetic inputs explicitly.

In [13]:
historical_timing = pd.DataFrame([{'N': 128, 'R': 64, 'states': 4225, 'calls': 23, 'seconds': 0.022409, 'table_MiB': 0.032234, 'ratio_lower': 1.272563, 'ratio_upper': 1.272564}, {'N': 256, 'R': 128, 'states': 16641, 'calls': 23, 'seconds': 0.046397, 'table_MiB': 0.126961, 'ratio_lower': 1.268713, 'ratio_upper': 1.268713}, {'N': 512, 'R': 256, 'states': 66049, 'calls': 23, 'seconds': 0.108933, 'table_MiB': 0.503914, 'ratio_lower': 1.294723, 'ratio_upper': 1.294723}, {'N': 896, 'R': 448, 'states': 201601, 'calls': 22, 'seconds': 0.20445, 'table_MiB': 1.538094, 'ratio_lower': 1.287227, 'ratio_upper': 1.287227}, {'N': 2048, 'R': 1024, 'states': 1050625, 'calls': 23, 'seconds': 0.634617, 'table_MiB': 8.015633, 'ratio_lower': 1.298713, 'ratio_upper': 1.298713}, {'N': 4864, 'R': 2432, 'states': 5919489, 'calls': 21, 'seconds': 2.194086, 'table_MiB': 45.162117, 'ratio_lower': 1.293959, 'ratio_upper': 1.293959}])
# Saved outputs of notebook 02 at base f3620ce; not new timings.
historical_timing[["N", "R", "calls", "seconds", "table_MiB"]]

,N,R,calls,seconds,table_MiB
0,128,64,23,0.022409,0.032234
1,256,128,23,0.046397,0.126961
2,512,256,23,0.108933,0.503914
3,896,448,22,0.204450,1.538094
4,2048,1024,23,0.634617,8.015633
5,4864,2432,21,2.194086,45.162117


In [14]:
def call_bound(s, b1, b2):
    kappa = max(1., costs(1,s,b1,b2)[1]/b2)
    return int(np.ceil(np.log2(kappa/RTOL))) + 2

reference = historical_timing.set_index("N").loc[4864]
seconds_per_state_call = reference.seconds/(reference.states*reference.calls)
plans = []
for row in shape_summary.to_dict("records"):
    n, q = int(row["n"]), float(row["KiB/row"])
    calls = call_bound(s_kib/q,q*c1,q*(c1+delta))
    states = sum((int(np.floor(f*n))+1)*(n-int(np.floor(f*n))+1) for f in BUDGETS)
    plans.append(dict(shape=row["shape"], cases=int(row["traces"])*len(BUDGETS),
        projected_oracle_minutes=seconds_per_state_call*calls*states*row["traces"]/60,
        max_traceback_MiB=max(8*(int(np.floor(f*n))+1)*(n-int(np.floor(f*n))+1)/2**20 for f in BUDGETS)))
plan = pd.DataFrame(plans)
display(plan.round(3))
display(Markdown(f"Projected oracle-only work on the old host constant: **{plan.projected_oracle_minutes.sum():.1f} minutes**. This is not a new timing measurement."))

,shape,cases,projected_oracle_minutes,max_traceback_MiB
0,4864x896,648,23.176,45.162
1,896x128,1296,1.694,1.538
2,896x4864,1296,1.411,1.538
3,896x896,1296,1.581,1.538


Projected oracle-only work on the old host constant: **27.9 minutes**. This is not a new timing measurement.

## Explicit execution is resumable; ordinary evaluation is read-only.

Set `RUN_ORACLE = True` to compute missing cases. Native selector initialization is outside the per-case timers. Each completed case stores its masks, realized budget, objective bracket, diagnostics and runtime metadata atomically. Interrupting one case does not discard earlier cases.

The cache namespace hashes the CSV, full-precision model, trace archive, upstream revision, numerical recipe, selected traces and experiment settings. A changed input or algorithm starts a different namespace. Existing matching cases are reused; there is no automatic fallback to synthetic data. A normal evaluation reports missing cases without starting the sweep. The progress display is transient and is outside the measured solver calls.

In [15]:
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", dir=path.parent, delete=False, encoding="utf-8") as f:
        json.dump(value, f, allow_nan=False)
    Path(f.name).replace(path)


def check_payload(payload, trace, fraction):
    assert payload["fingerprint"] == run_id
    assert payload["trace_id"] == trace["trace_id"] and payload["fraction"] == float(fraction)
    assert payload["shape"] == trace["shape"]
    if payload["status"] != "ok":
        assert payload["status"] in {"empty_selection", "zero_importance"} and not payload["records"]
        return
    v = importance_vector(trace)
    q = weight_bytes*trace["d"]/1024
    T = costs(len(v),s_kib/q,q*c1,q*(c1+delta))
    assert len(payload["records"]) == 4
    for row in payload["records"]:
        indices = payload["masks"][row["method"]]
        assert len(indices) == len(set(indices)) == payload["R"]
        assert all(isinstance(i,int) and 0<=i<len(v) for i in indices)
        mask = np.zeros(len(v),bool)
        mask[indices] = True
        I,L = metrics(v,mask,T)
        np.testing.assert_allclose([I,L,I/L], [row["importance"],row["fitted_ms"],row["ratio"]], rtol=1e-10)

notebook = json.loads((HERE / "02_global_oracle.ipynb").read_text())
recipe = "\n".join("".join(c["source"]) for c in notebook["cells"]
    if "recipe" in c.get("metadata",{}).get("tags",[]))
request = dict(version="measured-fixed-r-v1", profile_sha256=sha256(PROFILE),
    model_sha256=sha256(MODEL), trace_sha256=sha256(TRACE), upstream=UPSTREAM_SHA,
    recipe_sha256=hashlib.sha256(recipe.encode()).hexdigest(), budgets=list(BUDGETS),
    selected_trace_ids=[int(t["trace_id"]) for t in traces], rtol=RTOL,
    paper_impl=PAPER_IMPL, geometry=PAPER_GEOMETRY, weight_bytes=weight_bytes,
    max_table_mib=MAX_TABLE_MIB, numpy=np.__version__, scipy=scipy.__version__)
run_id = hashlib.sha256(json.dumps(request,sort_keys=True).encode()).hexdigest()
run_dir = HERE / "oracle_runs" / run_id
cases = [(t,f) for t in traces for f in BUDGETS]


def case_path(trace, fraction):
    return run_dir / f"trace-{trace['trace_id']:06d}-budget-{BUDGETS.index(fraction):02d}.json"

if RUN_ORACLE:
    backend = load_upstream()
    atomic_json(run_dir / "manifest.json", dict(request=request, runtime=backend[3]))
    progress = display(Markdown("Computing the real-trace comparison…"), display_id=True)
    try:
        for index,(trace,fraction) in enumerate(cases,1):
            path = case_path(trace,fraction)
            if path.is_file():
                check_payload(json.loads(path.read_text()),trace,fraction)
            else:
                payload = dict(one_case(trace,fraction,backend),fingerprint=run_id,runtime=backend[3])
                check_payload(payload,trace,fraction)
                atomic_json(path,payload)
            progress.update(Markdown(f"Global oracle: **{index}/{len(cases)}** cases completed or reused."))
    finally:
        progress.update(Markdown(""))

completed = []
for trace,fraction in cases:
    path = case_path(trace,fraction)
    if path.is_file():
        payload = json.loads(path.read_text())
        check_payload(payload,trace,fraction)
        completed.append(payload)
frame = pd.DataFrame([r for p in completed for r in p["records"]])
exclusions = pd.DataFrame([{k:p[k] for k in ("trace_id","shape","fraction","status")}
                           for p in completed if p["status"] != "ok"])
display(Markdown(f"**{len(completed)}/{len(cases)} cases available.** " +
    ("The real-trace sweep is complete." if len(completed)==len(cases) else
     "The sweep is incomplete; no claim below represents the full archive.")))
if not exclusions.empty:
    display(exclusions)

**0/4536 cases available.** The sweep is incomplete; no claim below represents the full archive.

## Report shape-specific gaps, not one pooled verdict.

A larger importance/latency ratio need not mean more retained importance. The figures below appear only when matching real results exist. Medians and 90th percentiles are descriptive over the selected trace/budget cases. Prompt-level summaries retain the archive's actual grouping; they are not confidence intervals.

In [16]:
if not frame.empty:
    summary = frame.groupby(["shape","method"]).agg(
        cases=("trace_id","size"), median_gap_pct=("gap_upper_pct","median"),
        p90_gap_pct=("gap_upper_pct",lambda x: x.quantile(.9)),
        retained_pct=("retained_pct","median"), fitted_ms=("fitted_ms","median"),
        chunks=("chunks","median"), selector_seconds=("selector_seconds","median"),
        lookup_supported_pct=("lookup_supported",lambda x: 100*x.mean()))
    display(summary.round(4))
    gaps = frame[frame.method != "Global DP"].groupby(["shape","method"]).gap_upper_pct.median().unstack()
    ax = gaps.plot.bar(figsize=(9,3.6),rot=0)
    ax.set(xlabel="Projection shape (input rows × output columns)", ylabel="Median fitted-ratio gap upper bound (%)")
    plt.tight_layout(); plt.show()
    display(Markdown("**Figure 1.** Each method uses the upstream mask's realized row count. "
        "Gaps refer to the fitted additive objective, not device latency."))
    by_prompt = frame.groupby(["prompt","shape","method"]).gap_upper_pct.median().unstack()
    display(by_prompt.round(4))
else:
    display(Markdown("No real-trace comparison has been computed for this input fingerprint. The protocol is ready; results are not fabricated from the previous synthetic examples."))

No real-trace comparison has been computed for this input fingerprint. The protocol is ready; results are not fabricated from the previous synthetic examples.

In [17]:
if not frame.empty:
    for shape, group in frame[frame.method != "Global DP"].groupby("shape"):
        curve = group.groupby(["fraction","method"]).gap_upper_pct.median().unstack()
        ax = curve.plot(figsize=(8,3),marker=".")
        ax.set(title=f"{shape}: median fitted-objective gap", xlabel="Nominal retained fraction", ylabel="Gap upper bound (%)")
        plt.tight_layout(); plt.show()
    fill = frame[frame.method == "Upstream greedy"].groupby(["shape","fraction"]).fill_pct.agg(["min","median"])
    display(fill.round(3))

In [18]:
if not frame.empty:
    example = next((p for p in completed if p["status"]=="ok" and np.isclose(p["fraction"],.5)), None)
    if example is not None:
        rows = pd.DataFrame(example["records"])
        plt.figure(figsize=(8,3.4))
        for _,row in rows.iterrows():
            plt.scatter(row.fitted_ms,row.retained_pct,label=row.method)
        plt.xlabel("Fitted additive latency (ms)"); plt.ylabel("Retained importance (%)")
        plt.legend(fontsize=8); plt.tight_layout(); plt.show()
        display(rows.set_index("method")[["importance","fitted_ms","chunks","gap_upper_pct"]].round(4))
        n = next(t["n"] for t in traces if t["trace_id"]==example["trace_id"])
        geometry = np.zeros((len(example["masks"]),n),bool)
        for i,indices in enumerate(example["masks"].values()):
            geometry[i,indices] = True
        plt.figure(figsize=(9,2.4)); plt.imshow(geometry,aspect="auto",interpolation="nearest")
        plt.yticks(range(len(geometry)),list(example["masks"]))
        plt.xlabel(f"Trace {example['trace_id']}: channel; every mask has R={example['R']}")
        plt.tight_layout(); plt.show()

In [19]:
if not frame.empty:
    keys = ["trace_id","fraction","shape"]
    oracle = frame[frame.method=="Global DP"][keys+["lookup_ratio"]]
    paper = frame[frame.method=="Upstream greedy"][keys+["lookup_ratio"]]
    pairs = oracle.merge(paper,on=keys,suffixes=("_oracle","_paper"))
    pairs["paired_supported"] = pairs[["lookup_ratio_oracle","lookup_ratio_paper"]].notna().all(axis=1)
    valid = pairs[pairs.paired_supported].copy()
    valid["lookup_ratio_gain_pct"] = 100*(valid.lookup_ratio_oracle/valid.lookup_ratio_paper-1)
    display(pairs.groupby("shape").paired_supported.agg(["sum","count"]))
    if not valid.empty:
        ax = valid.groupby("shape").lookup_ratio_gain_pct.median().plot.bar(figsize=(8,3.2),rot=0)
        ax.set(xlabel="Projection shape (fully supported pairs only)",ylabel="Median held-out lookup ratio gain (%)")
        plt.tight_layout(); plt.show()
    display(Markdown("**Lookup robustness.** Unsupported pairs are excluded and counted, not extrapolated. "
        "A positive gain is not a proof of lookup-optimality or a measured wall-time speedup."))

## Large scaling is a separate, explicitly synthetic timing study.

At $R=N/2$, the largest planned value table has $(R+1)(N-R+1)$ float64 entries. The table below shows its allocation and the old-host state-work projection. Set `RUN_SCALING = True` to measure three complete ratio solves per size, including final recovery, on a seeded lognormal vector. It does not load a model or measure SSD I/O. Its row width is fixed at 1.75 KiB only to give a declared cost; dimensions absent from the archive are never presented as captured activations.

Scaling results have their own fingerprint and resume independently of the real-trace experiment. Timings are for this Python/SciPy oracle, not a sub-two-millisecond online implementation.

In [20]:
q = 1.75
s, b1, b2 = s_kib/q, q*c1, q*(c1+delta)
scaling_plan = pd.DataFrame([dict(N=n,R=n//2,states=(n//2+1)*(n-n//2+1),
    table_MiB=8*(n//2+1)*(n-n//2+1)/2**20,
    projected_seconds=seconds_per_state_call*call_bound(s,b1,b2)*(n//2+1)*(n-n//2+1))
    for n in SCALING_SIZES])
display(scaling_plan.round(3))
scaling_id = hashlib.sha256(json.dumps(dict(run_id=run_id,row_kib=q,repeats=SCALING_REPEATS,
                                          sizes=list(SCALING_SIZES))).encode()).hexdigest()
scaling_dir = HERE / "scaling_runs" / scaling_id
if RUN_SCALING:
    for n in SCALING_SIZES:
        path = scaling_dir / f"{n}.json"
        if path.is_file():
            continue
        v = np.random.default_rng(0).lognormal(0,1,n)
        v /= v.mean()
        seconds, counts = [], []
        for _ in range(SCALING_REPEATS):
            start = perf_counter()
            mask,lower,upper,calls = solve(v,n//2,s,b1,b2,RTOL,MAX_TABLE_MIB)
            seconds.append(perf_counter()-start); counts.append(calls)
        atomic_json(path,dict(fingerprint=scaling_id,N=n,R=n//2,seconds=seconds,calls=counts,
            platform=platform.platform(),python=sys.version,numpy=np.__version__,scipy=scipy.__version__))
scaling_rows = []
for n in SCALING_SIZES:
    path = scaling_dir / f"{n}.json"
    if path.is_file():
        result = json.loads(path.read_text())
        assert result["fingerprint"] == scaling_id and result["N"] == n
        assert len(result["seconds"]) == SCALING_REPEATS
        scaling_rows.append(dict(N=n,seconds=np.median(result["seconds"]),calls=np.median(result["calls"])))
if scaling_rows:
    timing = pd.DataFrame(scaling_rows)
    plt.figure(figsize=(8,3.5))
    plt.loglog(timing.N,timing.seconds,"o-",label="New measured solves (median)")
    plt.xlabel("Channels N"); plt.ylabel("Complete oracle solve (seconds)")
    plt.legend(); plt.tight_layout(); plt.show()
else:
    display(Markdown("The new large-size timing sweep has not been run. The table above contains projections only."))

,N,R,states,table_MiB,projected_seconds
0,128,64,4225,0.032,0.002
1,256,128,16641,0.127,0.008
2,512,256,66049,0.504,0.033
3,896,448,201601,1.538,0.100
4,2048,1024,1050625,8.016,0.519
5,4864,2432,5919489,45.162,2.925
6,8960,4480,20079361,153.193,9.923
7,14336,7168,51394561,392.109,25.400
8,18944,9472,89737729,684.645,44.349


The new large-size timing sweep has not been run. The table above contains projections only.

## The empirical conclusion remains conditional.

The oracle is global over masks of the realized cardinality under the inherited two-line cost. It is not global for an arbitrary lookup table, a coverage constraint, an entire Pareto frontier, or an end-to-end accuracy objective. The archive's mean absolute activation is an importance proxy, not a measurement of output error.

The default use of real traces does not remove the need for controlled examples: zero/tied inputs and short-run counterexamples remain in the exhaustive checks. Nor does a reproduced profile prove the additive model at unmeasured run lengths. Report lookup coverage, underfilling, case completeness and the number of actual prompts alongside any advantage.

The distinction is deliberate: notebook 01 estimates a cost; this notebook measures selection quality **under that cost**. Hardware mask latency and task-level accuracy are subsequent questions.